# 🤖 Notebook 5 — BERT Fine-Tuning
Fine-tune `bert-base-uncased` on AG News for text classification.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from src.feature_engineering import TextDataset
from config import BERT_MODEL_NAME, MAX_LEN, BATCH_SIZE, EPOCHS, LEARNING_RATE, MODEL_DIR
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load data (use a subset for speed on CPU)
train_df = pd.read_csv('../data/processed/train_clean.csv').dropna(subset=['clean_text'])
test_df  = pd.read_csv('../data/processed/test_clean.csv').dropna(subset=['clean_text'])

# Use 5000 / 1000 samples for quick demo (remove slicing for full training)
train_df = train_df.sample(5000, random_state=42)
test_df  = test_df.sample(1000, random_state=42)
print('Train:', len(train_df), ' Test:', len(test_df))

In [ ]:
# Tokenize
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

def tokenize(texts, max_len=MAX_LEN):
    return tokenizer(list(texts), padding=True, truncation=True,
                     max_length=max_len, return_tensors='pt')

train_enc = tokenize(train_df['clean_text'])
test_enc  = tokenize(test_df['clean_text'])

train_ds = TextDataset(train_enc, train_df['label'].reset_index(drop=True))
test_ds  = TextDataset(test_enc,  test_df['label'].reset_index(drop=True))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)
print('✅ Tokenization complete!')

In [ ]:
# Load model
model = BertForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=4)
model.to(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
print('✅ BERT loaded!')

In [ ]:
# Train
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}')

In [ ]:
# Evaluate
model.eval()
correct = total = 0
with torch.no_grad():
    for batch in test_loader:
        batch  = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop('labels')
        preds  = model(**batch).logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
print(f'🎯 BERT Accuracy: {correct/total:.4f}')

In [ ]:
# Save
save_path = os.path.join('..', MODEL_DIR if not MODEL_DIR.startswith('/') else '', 'bert_model')
save_path = '../saved_models/bert_model'
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f'💾 BERT saved → {save_path}')